# Rozdział 1 — Wprowadzenie do uczenia ze wzmocnieniem (wersja pełna)

Lektura: Sutton & Barto, rozdz. 1.

W tym notebooku:
- liczysz return $G_t$ (z dyskontem $\gamma$)
- estymujesz $v_\pi(s)$ z danych (roll-outy w FrozenLake)


## Interfejs agent–środowisko

![Agent–environment loop](../Images/agent_environment.png)

Ta pętla pojawia się w każdym algorytmie:
- obserwuj stan $S_t$
- wybierz akcję $A_t$
- odbierz nagrodę $R_{t+1}$ i następny stan $S_{t+1}$


In [1]:
from pathlib import Path
import sys

def find_project_root(marker: str = 'utils') -> Path:
    """Find the project root by searching for a marker folder/file.

    Works for classic Jupyter, VSCode notebooks, and many hosted notebook
    environments (Colab/Kaggle) where the current working directory may not be
    the notebook's directory.
    """
    cwd = Path.cwd().resolve()

    # Search upward (cwd, parent, grandparent, ...)
    for p in [cwd] + list(cwd.parents):
        if (p / marker).exists():
            return p

    # If cwd is something like /content, try a shallow downward search
    for p in cwd.glob('*'):
        if p.is_dir() and (p / marker).exists():
            return p
        if p.is_dir():
            for q in p.glob('*'):
                if q.is_dir() and (q / marker).exists():
                    return q

    return cwd

ROOT = find_project_root('utils')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

import gymnasium as gym
from utils import JupyterRender

%matplotlib inline

def make_frozenlake(is_slippery: bool = False, render_mode: str = "rgb_array"):
    return gym.make("FrozenLake-v1", is_slippery=is_slippery, render_mode=render_mode)

env = make_frozenlake(is_slippery=False, render_mode="rgb_array")
obs, info = env.reset(seed=0)
obs


0

## Epizody i return

FrozenLake jest zadaniem epizodycznym. Zdyskontowany return od chwili $t$:

$$G_t = \sum_{k=0}^{T-t-1} \gamma^k R_{t+1+k}$$

gdzie $\gamma \in [0,1]$ to współczynnik dyskonta, a $T$ to czas zakończenia epizodu.


## Ćwiczenie 1 — implementacja zdyskontowanego return

Zaimplementuj `discounted_return(rewards, gamma)`.

Wymagania:
- wejście: lista/tablica nagród `[R_1, R_2, ..., R_T]`
- wyjście: skalar $G_0$
- uwzględnij dyskonto $\gamma$


> **Notatki dla prowadzącego:**  
> - Return od czasu $t=0$: $G_0 = \sum_{k=0}^{T-1}\gamma^k R_{k+1}$.  
> - Najprościej policzyć iteracyjnie „od końca”: `G=0; for r in reversed(rewards): G = r + gamma*G`.  
> - W FrozenLake nagrody są $0/1$, więc $G_0$ często jest 0 (brak celu) albo blisko 1 (sukces).  
> - Uwaga na indeksowanie: w liście są nagrody $R_1,\dots,R_T$ (bez $R_0$).

In [6]:
from typing import Sequence

def discounted_return(rewards: Sequence[float], gamma: float) -> float:
    G = 0.0
    pow_gamma = 1.0
    for r in rewards:
        G += pow_gamma * float(r)
        pow_gamma *= gamma
    return float(G)

# tests
assert abs(discounted_return([1.0], gamma=0.9) - 1.0) < 1e-9
assert abs(discounted_return([0.0, 1.0], gamma=0.9) - 0.9) < 1e-9
print("Exercise 1 tests passed ✅")


Exercise 1 tests passed ✅


## Polityki

Deterministyczną politykę będziemy reprezentować tablicą:

- `pi[s]` to akcja (0..3) wykonywana w stanie `s`.


In [7]:
n_states = env.observation_space.n
n_actions = env.action_space.n

# Example deterministic policy (all "right")
pi_right = np.full(n_states, 2, dtype=int)  # 2 = right

def policy_from_array(pi):
    return lambda s: int(pi[s])

# quick rollout
rng = np.random.default_rng(0)

def rollout_episode(env, policy, rng, max_steps=100):
    s, info = env.reset(seed=int(rng.integers(0, 1_000_000)))
    rewards = []
    for _ in range(max_steps):
        a = int(policy(s))
        s, r, terminated, truncated, info = env.step(a)
        rewards.append(r)
        if terminated or truncated:
            break
    return rewards

rewards = rollout_episode(env, policy_from_array(pi_right), rng)
rewards, discounted_return(rewards, gamma=0.99)


([0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0],
 0.0)

## Funkcje wartości

Najczęściej używane:
- wartość stanu: $$v_\pi(s) = \mathbb{E}_\pi[G_t \mid S_t=s]$$
- wartość pary stan–akcja: $$q_\pi(s,a) = \mathbb{E}_\pi[G_t \mid S_t=s, A_t=a]$$

W tym rozdziale skupiamy się na $v_\pi$.


## Ćwiczenie 2 — estymacja Monte Carlo $v_\pi(s)$

Zaimplementuj `mc_estimate_v(env, pi, episodes, gamma)`:

- każdy epizod zaczynaj od `env.reset()`
- podążaj za polityką `pi`
- policz return dla odwiedzonych stanów
- uśrednij po epizodach

Wskazówka: w FrozenLake możesz zacząć od estymacji wartości stanu startowego.


> **Notatki dla prowadzącego:**  
> - Monte Carlo: generujemy epizody polityką $\pi$, liczymy returny $G_t$ i uśredniamy po odwiedzinach stanu.  
> - Tabular implementacja: wektory `V[s]` i `N[s]`; aktualizacja przyrostowa `V[s] += (G - V[s]) / N[s]`.  
> - Warto wspomnieć różnicę *first-visit* vs *every-visit* (tu można użyć każdej wizyty).  
> - Jeśli wyniki są bliskie 0: to może być poprawne (rzadkie sukcesy); zwiększ `episodes` albo użyj `is_slippery=False` na start.

In [8]:
def mc_value_start_state(env, pi: np.ndarray, gamma: float, episodes: int = 5000, seed: int = 0) -> dict:
    rng = np.random.default_rng(seed)
    policy = lambda s: int(pi[s])
    returns = []
    successes = 0
    for _ in range(episodes):
        rewards = rollout_episode(env, policy, rng)
        G = discounted_return(rewards, gamma=gamma)
        returns.append(G)
        successes += int(np.sum(rewards) > 0.0)
    return {
        "v_start_est": float(np.mean(returns)),
        "success_rate": successes / episodes,
        "return_std": float(np.std(returns)),
    }

# Random-ish policy
pi_randomish = np.array([0,1,2,3] * (n_states // 4), dtype=int)
out = mc_value_start_state(env, pi_randomish, gamma=0.99, episodes=2000, seed=0)
out


{'v_start_est': 0.0, 'success_rate': 0.0, 'return_std': 0.0}

## Dlaczego to działa

Monte Carlo (MC) estymuje wartość oczekiwaną przez uśrednianie wielu próbek returnu.
Dla ustalonej polityki $\pi$ i stanu $s$:

- każda próba returnu ma wartość oczekiwaną $v_\pi(s)$
- średnia z wielu epizodów zbiega do $v_\pi(s)$ (prawo wielkich liczb)

Wadą MC jest duża wariancja. W kolejnych rozdziałach zobaczysz metody TD, które bootstrapują.


## Podsumowanie

- Return $G$ definiuje, co maksymalizuje agent.
- $v_\pi$ opisuje „jak dobrze jest” w stanie, gdy podążamy za polityką $\pi$.

Następnie: Rozdział 2 (bandici) i Rozdział 3 (MDP).
